# Process A: stratified train/test split

* The split is done **per batch**, not per row. All time steps of a batch go to the same set, so no batch appears in both training and test (this avoids data leakage).
* Test set = 30% of the batches, `random_state=42`.
* **Stratified by label:** the accepted/rejected ratio (about 75% / 25%) is the same in the training and test sets, so the under-represented `rejected` class is represented proportionally in both.
* The input files are only read. The output CSVs contain the original rows plus a `label` column.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = "../input-data/Project 2 - BRPMM/process A/calibration data/"
SEED = 42
TEST_SIZE = 0.30

df = pd.read_csv(DATA_DIR + "calibration_data.csv")
labels = pd.read_csv(DATA_DIR + "calibration_labels.csv")
print(df.shape, "| batches:", df.batch_id.nunique())
print(labels.label.value_counts())

(415165, 12) | batches: 161
label
accepted    121
rejected     40
Name: count, dtype: int64


In [2]:
train_ids, test_ids = train_test_split(
    labels.batch_id,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=labels.label,      # keep class proportions identical in both sets
)
assert set(train_ids).isdisjoint(test_ids)

data = df.merge(labels, on="batch_id", how="left")   # add the label to every row
train = data[data.batch_id.isin(train_ids)].reset_index(drop=True)
test = data[data.batch_id.isin(test_ids)].reset_index(drop=True)

In [3]:
def class_table(ids):
    lab = labels.set_index("batch_id").loc[ids, "label"]
    return pd.DataFrame({"batches": lab.value_counts(), "share": lab.value_counts(normalize=True).round(3)})

print("All batches:"); display(class_table(labels.batch_id))
print("Training set:"); display(class_table(train_ids))
print("Test set:"); display(class_table(test_ids))
print(f"Rows -> train: {len(train):,}  test: {len(test):,}  "
      f"(batches: {len(train_ids)} / {len(test_ids)}, test fraction = {len(test_ids)/len(labels):.1%})")

All batches:


,batches,share
label,,
accepted,121,0.752
rejected,40,0.248


Training set:


,batches,share
label,,
accepted,84,0.75
rejected,28,0.25


Test set:


,batches,share
label,,
accepted,37,0.755
rejected,12,0.245


Rows -> train: 290,496  test: 124,669  (batches: 112 / 49, test fraction = 30.4%)


In [4]:
train.to_csv("training_process_A.csv", index=False)
test.to_csv("test_process_A.csv", index=False)
print("Saved training_process_A.csv and test_process_A.csv")

Saved training_process_A.csv and test_process_A.csv
